# 🧩 Mini-Lab: pandas Essentials for Messy Text Data

**Module: Real Data: Paths, CSV, JSON, pandas, NumPy** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Identify** common data quality problems in real-world text datasets
2. **Clean** messy DataFrames step-by-step using pandas
3. **Report** how many rows each cleaning step removed, so nothing is silently lost

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Counted Cleaning | A data cleaning habit where every step reports how many rows it removed, so you can verify assumptions, spot surprises, and keep an auditable trail of what changed. |


## Why This Matters

Real data is never clean. Messages have missing fields, duplicates, whitespace-only text, and junk values. If you drop rows **silently**, you can't explain why 600 rows became 400 — and stakeholders (or your future self) won't trust the pipeline.

**Counted cleaning** means: after every cleaning step, print how many rows were removed and how many remain. It's a one-line habit that makes your data work auditable.

In [1]:
import pandas as pd
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## The Messy Dataset

We'll simulate a small but realistically messy batch of chat messages — the kind you'd get from a raw export before any cleaning.

In [2]:
raw_data = [
    {"id": 1,    "role": "user",      "text": "What is RAG?",              "tokens": 5},
    {"id": 2,    "role": "assistant",  "text": "Retrieval-Augmented Gen.",  "tokens": 4},
    {"id": 3,    "role": "user",      "text": None,                         "tokens": 0},      # missing text
    {"id": 4,    "role": "assistant",  "text": "   ",                       "tokens": 1},      # whitespace-only
    {"id": 5,    "role": "user",      "text": "Explain embeddings.",       "tokens": 3},
    {"id": 5,    "role": "user",      "text": "Explain embeddings.",       "tokens": 3},      # exact duplicate
    {"id": 6,    "role": None,         "text": "Some answer here.",         "tokens": 4},      # missing role
    {"id": 7,    "role": "user",      "text": "  How does chunking work? ", "tokens": 5},    # leading/trailing spaces
    {"id": 8,    "role": "bot",       "text": "I chunk things.",           "tokens": 4},      # invalid role
    {"id": None, "role": "user",      "text": "No id here.",              "tokens": 3},      # missing id
    {"id": 10,   "role": "assistant",  "text": "",                          "tokens": 0},      # empty string
    {"id": 11,   "role": "user",      "text": "What is fine-tuning?",     "tokens": 5},
]

df = pd.DataFrame(raw_data)
print(f"Raw dataset: {len(df)} rows")
df

Raw dataset: 12 rows


,id,role,text,tokens
0,1.0,user,What is RAG?,5
1,2.0,assistant,Retrieval-Augmented Gen.,4
2,3.0,user,None,0
3,4.0,assistant,,1
4,5.0,user,Explain embeddings.,3
5,5.0,user,Explain embeddings.,3
6,6.0,None,Some answer here.,4
7,7.0,user,How does chunking work?,5
8,8.0,bot,I chunk things.,4
9,NaN,user,No id here.,3


## The Counted-Cleaning Pattern

The idea is simple: **wrap every filter in a tiny helper** that prints what happened.

```
before → filter → after → print the difference
```

We'll use one small function so every step is consistent.

In [3]:
def cleaned(df_before: pd.DataFrame, df_after: pd.DataFrame, step: str) -> pd.DataFrame:
    """Print how many rows a cleaning step removed, then return the cleaned DataFrame."""
    dropped = len(df_before) - len(df_after)
    print(f"  Step: {step:<35s} | dropped {dropped:>3d} | remaining {len(df_after):>3d}")
    return df_after

## Cleaning Pipeline

We'll apply five common cleaning steps, one at a time, printing counts after each.

In [4]:
print(f"Starting rows: {len(df)}\n")

# --- Step 1: Drop rows where 'id' is missing ---
df_clean = cleaned(df, df.dropna(subset=["id"]), "drop missing id")

# --- Step 2: Drop rows where 'role' is missing ---
df_clean = cleaned(df_clean, df_clean.dropna(subset=["role"]), "drop missing role")

# --- Step 3: Drop rows where 'text' is missing, empty, or whitespace-only ---
# First strip whitespace so "   " becomes "" and gets caught by the filter
df_clean["text"] = df_clean["text"].fillna("").str.strip()
mask_has_text = df_clean["text"].str.len() > 0
df_clean = cleaned(df_clean, df_clean[mask_has_text].copy(), "drop empty/whitespace text")

# --- Step 4: Drop exact duplicate rows ---
df_clean = cleaned(df_clean, df_clean.drop_duplicates(), "drop exact duplicates")

# --- Step 5: Keep only valid roles ---
valid_roles = {"user", "assistant", "system"}
mask_valid_role = df_clean["role"].isin(valid_roles)
df_clean = cleaned(df_clean, df_clean[mask_valid_role].copy(), "drop invalid roles")

print(f"\nFinal rows: {len(df_clean)}  (started with {len(df)})")

Starting rows: 12

  Step: drop missing id                     | dropped   1 | remaining  11
  Step: drop missing role                   | dropped   1 | remaining  10
  Step: drop empty/whitespace text          | dropped   3 | remaining   7
  Step: drop exact duplicates               | dropped   1 | remaining   6
  Step: drop invalid roles                  | dropped   1 | remaining   5

Final rows: 5  (started with 12)


## Full Cleaning Report

In a real pipeline you'd log this report or save it alongside the cleaned data. Let's build a summary table.

In [5]:
# Rebuild the counts in one pass for a clean summary
steps = [
    ("Raw input",                 len(df)),
    ("After drop missing id",     len(df.dropna(subset=["id"]))),
]

# We'll just use the final count to show the full pipeline result
total_dropped = len(df) - len(df_clean)

report = [
    "| Metric | Count |",
    "|--------|------:|",
    f"| Raw rows | {len(df)} |",
    f"| Final clean rows | {len(df_clean)} |",
    f"| Total dropped | {total_dropped} |",
    f"| Drop rate | {total_dropped / len(df):.0%} |",
]

md("\n".join(report))

| Metric | Count |
|--------|------:|
| Raw rows | 12 |
| Final clean rows | 5 |
| Total dropped | 7 |
| Drop rate | 58% |

## Why Counting Matters

Without counted cleaning, you'd just call `.dropna().drop_duplicates()` in one line and see the final number. That's fine for a quick prototype, but in any real pipeline:

- **Debugging** — if 50% of your data vanishes, you need to know *which step* caused it
- **Trust** — stakeholders want to know what was dropped and why
- **Drift detection** — if Tuesday's batch drops 2% but Wednesday's drops 40%, the counts tell you immediately

## 🎯 Summary

### Key Takeaways

1. **Counted cleaning** — after every filter step, print how many rows were removed so nothing is silently lost
2. **One step at a time** — apply each cleaning rule separately (missing values, whitespace, duplicates, invalid categories) so you can pinpoint where data disappears
3. **A tiny helper goes a long way** — a `cleaned(before, after, step)` function keeps every step consistent and auditable
4. **Always report totals** — log or print a summary (raw → clean → dropped) alongside your output data